# Full agent eval: every tool, with memory

The other notebooks isolate one backend. This one is the whole agent — the same
`create_agent` instance the Discord bot runs, imported from `agent.agent`, with
all 10 read-only tools and the shared SQLite checkpointer. Questions asked on the
same `thread_id` remember each other.

Use it to watch tool _selection_ — which backend the model reaches for when
nothing is taken away — and whether it carries context forward.


In [ ]:
from eval_transcript import print_summary, run_agent_eval
from agent.tools import ALL_TOOLS, configured_tools

print(f"{len(ALL_TOOLS)} tools in the catalogue:", ", ".join(t.name for t in ALL_TOOLS))
print(f"{len(configured_tools())} actually bound:", ", ".join(t.name for t in configured_tools()))


### Reading a turn

The agent returns the whole thread; these pull the answer text and the token
cost out of the messages one call added.


`run_agent_eval` drives the production agent and writes the same transcript
shape the isolation notebooks do, so every run here lands in eval/logs/ and is
picked up by the same summariser. Only the messages one call adds are measured.


### Running one question

`thread_config` supplies the thread id, exactly as in the docs
(`{"configurable": {"thread_id": ...}}`). The tool budget, tool-error handling,
and context summarization are middleware on the agent itself, so nothing is
configured per call. Tokens are tallied over only the messages this call added,
so a follow-up on the same thread reports its own cost.


In [ ]:
THREAD = "notebook"
FRESH = "fresh"


## A broad question

Nothing is taken away, so the model picks its own path. Watch the printed tool
calls: does it start with `list_services`, and does it reach for discovery before
querying Prometheus or Loki?


In [ ]:
answer = await run_agent_eval(
    "08_all_tools:broad",
    "Is anything wrong in the lab right now?",
    thread_id=THREAD,
)
print(answer)


## A follow-up on the same thread

This is the memory test. The question names nothing — no container, no metric —
so it is only answerable if the checkpointed thread carried the previous turn
forward.


In [ ]:
answer = await run_agent_eval(
    "08_all_tools:followup-same-thread",
    "Why? Show me the evidence for that.",
    thread_id=THREAD,
    resume=True,  # memory is the thing under test here
)
print(answer)


## A clean thread

A different `thread_id` shares nothing, which is what keeps two Discord channels
apart. The same question that just worked should now fall flat.


In [ ]:
# The same question with no history. There's no automated check for "does it
# admit it has no context" — read the printed answer: passing looks like the
# agent asking what you mean, not inventing a prior turn.
answer = await run_agent_eval(
    "08_all_tools:followup-fresh-thread",
    "Why? Show me the evidence for that.",
    thread_id=FRESH,
)
print(answer)


## Starting over

There is no external API for dropping a thread — the docs define `thread_id` as
the only handle on a conversation. To re-run this notebook from a clean slate,
use a new `thread_id`, or delete `data/memory.db` while the bot is stopped.


## Collected numbers

Every run in this notebook and the seven isolation notebooks wrote a
transcript to `eval/logs/`. This reads them all back into one table.


In [ ]:
print_summary()
